<a href="https://colab.research.google.com/github/Dyuko/DataScience/blob/main/rag_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Alumno: Matías Irala

CI: 4637852

# Parte 1: Preparación del entorno

In [1]:
!pip install langchain
!pip install langchain-community
!pip install langchain-core
!pip install langchain-groq

In [2]:
!pip install faiss-cpu

In [3]:
import os

import dotenv
from google.colab import drive

drive.mount('/content/drive')

dotenv.load_dotenv('/content/drive/MyDrive/.env')

api_key = os.environ.get('GROP_API_KEY')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Parte 2: Construcción del sistema RAG

## 1. Carga de documentos

In [4]:
# Cargar csv EdSheeran.csv desde drive
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/datos/EdSheeran.csv').dropna(subset=["Lyric"])

In [5]:
#df.head()

In [6]:
from langchain_core.documents import Document
# Convertir cada fila en un Document de LangChain
pages = []
for _, row in df.iterrows():
    # El contenido principal será la letra (Lyric)
    content = row["Lyric"]

    # Metadatos (opcional, pero útil para filtrado)
    metadata = {
        "id": row["Id"],
        "artist": row["Artist"],
        "title": row["Title"],
        "album": row["Album"],
        "year": row["Year"],
        "date": row["Date"],
    }

    pages.append(Document(page_content=content, metadata=metadata))

In [7]:
#pages

## 2. División en fragmentos

Usa CharacterTextSplitter para dividir los documentos en chunks de 500 caracteres.

In [8]:
## Split and Store
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Initialize splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

## Initialize embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

## Split documents
documents = text_splitter.split_documents(pages)

<ipython-input-8-3e2e45d55ba2>:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.wa

In [9]:
#documents

## 3. Creación del índice semántico

Usa FAISS para construir un índice vectorial a partir de los chunks usando embeddings como OpenAIEmbeddings.

In [10]:
## Vectorize documents into vector store
from langchain_community.vectorstores import FAISS
vector = FAISS.from_documents(documents,  ## documents
                              embeddings) ## embedding model

## 4. Configuración del LLM

Usa OpenAIChat o ChatOpenAI si estás usando GPT-3.5 o (LLamaCpp o ollama) si estás trabajando localmente.

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

chat = ChatGroq(temperature=0.3, groq_api_key=api_key, model_name="llama3-70b-8192")

## 5. 5. Construcción del RAG chain

Crea un RetrievalQA chain que combine el retriever (índice FAISS) con el modelo generador (LLM).

In [12]:
# Definir el prompt template para el RAG

from langchain.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate

## define prompt template
template = """Eres un experto en analizar letras de canciones. Responde la pregunta del usuario
utilizando SOLO los siguientes fragmentos de letras. No uses conocimiento externo.

Si los fragmentos no contienen suficiente información para responder,
di: "No encuentro letras relevantes para responder esto."

Contexto (fragmentos de letras):
{context}

Pregunta: {question}

Respuesta (basada SOLO en las letras anteriores, incluyendo siempre el nombre de la canción cuando menciones letras):"""

prompt = ChatPromptTemplate.from_template(template)

In [13]:
qa_chain = RetrievalQA.from_chain_type(
    llm=chat,
    chain_type="stuff",
    retriever=vector.as_retriever(search_kwargs={"k": 5}),  # Reducido a 5 para respuestas más concisas
    chain_type_kwargs={
        "prompt": prompt,
        "document_prompt": ChatPromptTemplate.from_template(
            "Canción: {title}\nÁlbum: {album}\nAño: {year}\nFragmento: {page_content}"
        )
    },
    return_source_documents=True
)

## 6. Evaluación del sistema

Formula al menos 5 preguntas que solo puedan responderse con los documentos cargados.

Evalúa la calidad y precisión de las respuestas.

In [14]:
def ask_question(question):
    print(f"\n\033[1mPregunta:\033[0m {question}")
    result = qa_chain.invoke({"query": question})

    print("\n\033[1mRespuesta:\033[0m")
    print(result["result"])

    print("\n\033[1mFuentes relevantes:\033[0m")
    seen_songs = set()
    for i, doc in enumerate(result["source_documents"], 1):
        song_id = doc.metadata['title']
        if song_id not in seen_songs:
            seen_songs.add(song_id)
            print(f"\n\033[1m{i}. Canción:\033[0m {doc.metadata['title']}")
            print(f"   \033[1mÁlbum:\033[0m {doc.metadata['album']} ({doc.metadata['year']})")
            print(f"   \033[1mFragmento relevante:\033[0m {doc.page_content[:200]}...")

In [15]:
ask_question("Which Ed Sheeran songs are about love?")


Pregunta: Which Ed Sheeran songs are about love?

Respuesta:
Basándome solo en los fragmentos de letras proporcionados, puedo identificar las siguientes canciones de Ed Sheeran que tratan sobre el amor:

* "Summer Nights/No Love For The Lonely (Ed Demo)" - La letra habla sobre la búsqueda de amor y la advertencia de que su amor es "una vela vacilante en la oscuridad".
* "No Love for the Lonely" - La letra menciona la búsqueda de amor y la frustración de encontrar solo lujuria en lugar de amor verdadero.
* "Perfect (With Beyoncé - Live)" - La letra describe un amor encontrado y la promesa de no dejar ir a esa persona.
* "Summer Nights/No Love For The Lonely (Ed Demo)" (de nuevo, porque se menciona la búsqueda de amor y la advertencia de que su amor es "una vela vacilante en la oscuridad").

No encuentro letras relevantes en "Supermarket Flowers (Remix)" que indiquen que se trata de una canción sobre el amor.

Fuentes relevantes:

1. Canción: Summer Nights/No Love For The Lonely (Ed Dem

In [16]:
ask_question("Can you name any lyrics that talk about nostalgia?")


Pregunta: Can you name any lyrics that talk about nostalgia?

Respuesta:
Sí, puedo encontrar letras que hablan sobre nostalgia.

En la canción "The West Coast of Clare", se menciona "vivid memories fade but the mood still remains" y "i wish i could go back and be with you again", lo que sugiere un sentimiento de nostalgia y deseo de regresar a un momento pasado.

También en la misma canción, se habla de "memories i have of you won't leave me in peace" y "my mind was running back to the west coast of clare", lo que indica una fuerte conexión con el pasado y un deseo de revivir momentos pasados.

En la canción "You", se menciona "tell 'em while i'm here cause one day i'm gonna die and when i'm gone i want my music to play", lo que sugiere un deseo de dejar un legado y ser recordado en el futuro, lo que puede ser visto como una forma de nostalgia hacia el propio pasado y la huella que se deja en el mundo.

Fuentes relevantes:

1. Canción: Oh No
   Álbum: nan (nan)
   Fragmento relevante:

In [17]:
ask_question("What song mentions cigarette?")


Pregunta: What song mentions cigarette?

Respuesta:
Basándome solo en los fragmentos de letras proporcionados, no encuentro menciones explícitas a "cigarette" en ninguna de las canciones. Sin embargo, hay referencias a "smoke" en varias canciones, como "Wake Me Up", "Nothing on You", "You Need Me, I Don’t Need You (Live at the Bedford)" y "The City (Live at the Bedford)". 

En "You Need Me, I Don’t Need You (Live at the Bedford)", se menciona "burning weed" y "spliff", lo que sugiere el consumo de marihuana. En "Trap Queen (Fetty Wap Cover)", se menciona "smoking dope" y "backwoods", que también se refiere al consumo de marihuana. 

En resumen, no encuentro letras relevantes que mencionen específicamente "cigarette", pero hay referencias a "smoke" y consumo de marihuana en varias canciones.

Fuentes relevantes:

1. Canción: Wake Me Up
   Álbum: + (Plus) (2011.0)
   Fragmento relevante: of smoke you always try and get me to stop but you drink as much as me and i get drunk a lot so ill 

In [18]:
ask_question("What song mentions arise from my tomb ?")


Pregunta: What song mentions arise from my tomb ?

Respuesta:
La respuesta es: "You Need Me, I Don’t Need You" (Live Ustream Version) y "You Need Me, I Don’t Need You" (Live at The Live Room).

Ambas canciones mencionan la frase "arise from my tomb" en sus letras.

Fuentes relevantes:

1. Canción: Small Bump - Live From Wembley Stadium
   Álbum: nan (nan)
   Fragmento relevante: spoken  now wembley i've not played this song in a around two years maybe maybe longer but i feel like playing it today let's see how it goes   you're just a small bump unborn in four months you're br...

2. Canción: You Need Me, I Don’t Need You (Live Ustream Version)
   Álbum: You Need Me, I Don’t Need You (Remixes) (2011.0)
   Fragmento relevante: with my head in the sky ed sheeran urban angel coming ready to die so see the signs stand to the side open your eyes and take a look and realise the resurrection's arrived and as the mist clears i ari...

3. Canción: You Need Me, I Don’t Need You (Live at The Live

In [19]:
ask_question("Can you name any lyrics that talk about Nightmares?")


Pregunta: Can you name any lyrics that talk about Nightmares?

Respuesta:
What a great question!

Yes, I can name several lyrics that talk about Nightmares. In fact, the song "Nightmares" by Human is all about nightmares. Here are some examples:

* "I feel my nightmares watching me and when my dreams are sleeping I feel my nightmares watching me" (Nightmares - + Random Impulse + Sway + Wretch 32)
* "When the darkness creeps in I feel my nightmares watching me and when my dreams are sleeping I feel my nightmares" (Nightmares)
* "I'm mr hard worker in my nightmares I'm mr can't turn up and I missed my chance" (Nightmares)
* "I'm scared to sing cause my nightmares'll fade in" (Nightmares)
* "I feel my nightmares watching me and when my dreams are sleeping I feel my nightmares watching me oh oh oh I feel my nightmares watching me" (Nightmares - + Random Impulse + Sway + Wretch 32)

These lyrics all mention nightmares and the fear or anxiety that comes with them.

Fuentes relevantes:

1. C

Calidad y precisión de las respuestas:

* Las respuestas generadas muestran alta precisión cuando el contexto relevante es recuperado por el sistema de retrieval.
* Se observa coherencia semántica entre las preguntas formuladas y los fragmentos de letras proporcionados como evidencia.
* El principal desafío identificado radica en la efectividad del componente de retrieval. En el caso de la pregunta "What song mentions cigarette?", aunque la canción "Castle On The Hill (Acoustic)" contiene explícitamente la palabra clave en su letra, el sistema no la incluyó en el contexto proporcionado al LLM.